In [1]:
############# LIBRERÍAS #############

import matplotlib.pyplot as plt
%matplotlib inline

from braket.aws import AwsDevice
from braket.circuits import Circuit, Gate, Noise, Observable, Instruction
from braket.circuits.noise_model import (GateCriteria, NoiseModel,
                                         ObservableCriteria)
from braket.circuits.noises import (AmplitudeDamping, BitFlip, Depolarizing,
                                    PauliChannel, PhaseDamping, PhaseFlip,
                                    TwoQubitDepolarizing)
from braket.devices import LocalSimulator

import numpy as np
import pandas as pd

import itertools
from itertools import permutations

import os
from datetime import datetime


############# FUNCIONES #############

# Para guardar diccionarios
import json
def save_pet(pet,filename):
    with open(filename, 'w') as f:
        f.write(json.dumps(pet))
def load_pet(filename):
    with open(filename) as f:
        pet = json.loads(f.read())
    return pet

# Para generar proyectores de medición
def mediciones_locales(N):
    Px = [ ]
    for i in range(N):
        M = Instruction(Gate.H(), [i])
        C = Circuit([M])
        Px.append(C)

    Py = []
    for i in range(N):
        M1 = Instruction(Gate.Si(), [i]) 
        M2 = Instruction(Gate.H(), [i])      
        C1 = Circuit([M1,M2])
        Py.append(C1)

    Pz = []
    for i in range(N):
        M = Instruction(Gate.I(), [i])
        C = Circuit([M])
        Pz.append(C)
    return Px, Py, Pz

#Circuito GHZ
def generation_circuit_GHZ (N):
    circuit = Circuit()
    circuit.h(0)
    for i in range(1,N):
        circuit.cnot(0, i)
    return circuit

# Para armar lista de observables a medir
def observables(N, simetria):
    if simetria == 'permutacional':
        L0 = []
        for comb in itertools.combinations_with_replacement('IXYZ', N):
            l = ''.join(list(comb))
            m = list(l.strip(" "))
            L0.append(m) 
    return L0

# Función tomográfica
def tomografia(circuit,L0,noise_level,s,device):
    Tomos_Noises = []
    for x in range(len(noise_level)): # x es un diccionario de diccionarios
        noise_model=noise(noise_level[x])
        ### Estado real con ruido
        circ_0 = noise_model.apply(circuit)
        circ_0.density_matrix(target=list(range(N)))
        task_0 = device.run(circ_0, shots=0)
        result_0 = task_0.result()
        DM = result_0.values[0]
        np.save(carpeta_guardado+'/DM_'+ str(x)+'.npy',DM)   
        Mean_Values = {}    
        for M in L0:
            A = noise_model.apply(circuit)
            obs=eval('Observable.' + M[0] + '()')
            m=M[0]
            for i in np.arange(1,len(M)):
                obs=obs@eval('Observable.' + M[i] + '()')
                m=m+M[i]
            # Hacer mediciones que nos interesan
            A.expectation(observable = obs, target=list(range(N)))
            result = device.run(A, shots = s).result()
            Mean_Values[m] = result.values[0]
        Tomos_Noises.append(Mean_Values)
    return Tomos_Noises, DM


############# MODELO DE RUIDO #############
# FUNCIÓN
def noise(x):
    noise_model = NoiseModel()
    noise_model.add_noise(Depolarizing(x['Depolarizing_gateH']), GateCriteria(Gate.H))
    noise_model.add_noise(Depolarizing(x['Depolarizing_gateCNot']), GateCriteria(Gate.CNot))
    noise_model.add_noise(BitFlip(x['BitFlip_gateH']), GateCriteria(Gate.H))
    noise_model.add_noise(BitFlip(x['BitFlip_gateCNot']), GateCriteria(Gate.CNot))
    noise_model.add_noise(AmplitudeDamping(x['AmplitudeDamping_gateH']), GateCriteria(Gate.H))
    noise_model.add_noise(AmplitudeDamping(x['AmplitudeDamping_gateCNot']), GateCriteria(Gate.CNot))
    #noise_model.add_noise(PauliChannel(0.1, 0.2, 0.3), GateCriteria(gates=Gate.X, qubits=1))
    return noise_model

def noisy(tipo_ruido):
    noise_level=[]
    if tipo_ruido == 'Depolarizing':
        for i in range(len(lista)):
            noise_dic={}
            noise_dic['Depolarizing_gateH']= lista[i]/10
            noise_dic['Depolarizing_gateCNot']= lista[i]
            noise_dic['BitFlip_gateH']=listazeros[i]
            noise_dic['BitFlip_gateCNot']=listazeros[i]
            noise_dic['AmplitudeDamping_gateH']=listazeros[i]
            noise_dic['AmplitudeDamping_gateCNot']=listazeros[i]
            noise_level.append(noise_dic)
    elif tipo_ruido == 'BitFlip':
        for i in range(len(lista)):
            noise_dic={}
            noise_dic['Depolarizing_gateH']= listazeros[i]
            noise_dic['Depolarizing_gateCNot']= listazeros[i]
            noise_dic['BitFlip_gateH']=lista[i]/10
            noise_dic['BitFlip_gateCNot']=lista[i]
            noise_dic['AmplitudeDamping_gateH']=listazeros[i]
            noise_dic['AmplitudeDamping_gateCNot']=listazeros[i]
            noise_level.append(noise_dic)
    elif tipo_ruido == 'AmplitudeDamping':
        for i in range(len(lista)):
            noise_dic={}
            noise_dic['Depolarizing_gateH']= listazeros[i]
            noise_dic['Depolarizing_gateCNot']= listazeros[i]
            noise_dic['BitFlip_gateH']=listazeros[i]
            noise_dic['BitFlip_gateCNot']=listazeros[i]
            noise_dic['AmplitudeDamping_gateH']=lista[i]/10
            noise_dic['AmplitudeDamping_gateCNot']=lista[i]
            noise_level.append(noise_dic)
    return noise_level,noise_dic

In [3]:
############# TOMOGRAFÍA #############
print("leyendo entradas")
# ENTRADAS #
N=3 # Número de qubits
simetria= 'permutacional'
device = LocalSimulator("braket_dm")
c= 30 #número de mediciones
shots = [10,100,200,300,400,500,700,900,1000,1200]
tipos_ruido=['Depolarizing','BitFlip','AmplitudeDamping']
numero_noises=15#sin ruido poner =1
lista= np.linspace(0, 0.15, numero_noises)
listazeros=np.zeros(numero_noises)
#linspace(inicio, fin, numero muestras)

############# GENERADORES #############
estado='GHZ_'+str(N)+'qubits' #sólo para nombrar archivos
Px,Py,Pz = mediciones_locales(N) # Observables
L0=observables(N,simetria) # Base de mediciones
circuit=generation_circuit_GHZ(N)
circ_ideal=generation_circuit_GHZ(N) #para obtener rho ideal

print("Creando carpetas y readme")
# Creo carpeta para guardar resultados
carpeta_guardado_general = 'Mediciones_'+str(estado)
if not os.path.exists(carpeta_guardado_general):
    os.mkdir(carpeta_guardado_general)
    
f= open(carpeta_guardado_general+'/ENTRADAS_'+str(carpeta_guardado_general)+'.txt',"w+")
f.write(' '+'\r\n')
f.write('ENTRADAS SELECCIONADAS'+'\r\n')
f.write('Estado: '+ str(estado)+'\r\n')
f.write('Número de mediciones: '+str(c) +'\r\n')
f.write('Set_shots = ' + str(shots)+'\r\n')
f.write('Tipos de ruido = ' + str(tipos_ruido)+'\r\n')
f.write('Simetría: '+ str(simetria)+'\r\n')
f.write('Tamaño base de mediciones= '+ str(len(L0))+'\r\n')    
f.write(' '+'\r\n')
f.write(' ----------------------------------------------- \r\n')
f.write(' '+'\r\n')
f.write('ESPECIFICACIONES'+'\r\n')
f.write(' '+'\r\n')
f.write('Circuito:'+'\r\n')
f.write(str(circuit)+'\r\n')
f.write(' '+'\r\n')
f.write('Base de mediciones: '+ str(L0)+'\r\n')
f.write(' '+'\r\n')
f.write('Número de noises: ' + str(numero_noises) +'\r\n')
f.close()

ti = datetime.now()
for x in range(len(tipos_ruido)):
    tipo_ruido=tipos_ruido[x]
    print('Comenzando proceso para:',tipo_ruido)
    noise_level,noise_dic=noisy(tipo_ruido)

    carpeta_guardado=carpeta_guardado_general+('/')+str(estado)+('_')+str(tipo_ruido)
    if not os.path.exists(carpeta_guardado):
        os.mkdir(carpeta_guardado)

    save_pet(noise_level,carpeta_guardado+ '/noise_level' + '.txt')
    save_pet(shots, carpeta_guardado+ '/shots'+ '.txt')

    f= open(carpeta_guardado+'/ENTRADAS_'+str(tipo_ruido)+'.txt',"w+")
    f.write('Tipo de ruido = ' + str(tipo_ruido)+'\r\n')
    f.write('Número de noises: ' + str(numero_noises) +'\r\n')
    f.write('Noises usados: ' +'\r\n')
    for key in noise_dic:
        f.write(str(key)+(': '))
        for level in noise_level:
            f.write(str(level[key])+(' '))
        f.write('\r\n')
    f.close()
   
    # Guardo rho del estado sin ruido / ideal
    circ_ideal.density_matrix(target=list(range(N)))
    task_ideal = device.run(circ_ideal, shots=0)
    result_ideal = task_ideal.result()
    DM_ideal = result_ideal.values[0]
    np.save(carpeta_guardado+'/DM_ideal.npy',DM_ideal)

    for k in range(len(shots)):
        data_total = []
        for i in np.arange(1,c+1):
           data_bell = tomografia(circuit,L0,noise_level, shots[k],device)[0] # para distinto nivel de ruido
           #print(((i)*100)/len(np.arange(1,c+1)),'% completado') 
           data_total.append(data_bell)
        save_pet(data_total, carpeta_guardado+ '/'+ str(estado) + '_shots'+ str(shots[k]) + '.txt')
        print(round(((k+1)*100)/(len(shots)*len(tipos_ruido))+(x*100)/(len(tipos_ruido)),2),'%')
    print('Mediciones finalizadas para',tipo_ruido)
    print(' ')
    t = datetime.now()
    print('Tiempo procesando=', t-ti)
    print(' ')
print(' ')
print('Proceso total finalizado')


leyendo entradas
Creando carpetas y readme
Comenzando proceso para: Depolarizing


/home/astrolabio/.local/lib/python3.11/site-packages/braket/default_simulator/simulator.py:323: UserWarning: You are running a noise-free circuit on the density matrix simulator. Consider running this circuit on the state vector simulator: LocalSimulator("default") for a better user experience.
  warnings.warn(


33.33 %
Mediciones finalizadas para Depolarizing
 
Tiempo procesando= 0:00:22.441968
 
Comenzando proceso para: BitFlip
66.67 %
Mediciones finalizadas para BitFlip
 
Tiempo procesando= 0:00:45.107544
 
Comenzando proceso para: AmplitudeDamping
100.0 %
Mediciones finalizadas para AmplitudeDamping
 
Tiempo procesando= 0:01:10.295019
 
 
Proceso total finalizado
